## TP 5

### Successive Halving

Le successive halving est une technique qui s'associe aux méthodes GridSearch et RandomSearch. La [documentation](https://scikit-learn.org/stable/modules/grid_search.html#) donnée précédemment traite du successive halving.

**§ Expliquer en quoi consiste le successive halving**

Le successive halving est une méthode d'optimisation des hyperparamètres qui évalue d'abord un grand nombre de configurations avec peu de ressources. À chaque itération, les configurations les moins performantes sont éliminées et davantage de ressources sont attribuées aux configurations restantes. Cela permet de réduire le coût de calcul par rapport à une recherche exhaustive comme GridSearchCV.

**§ Expliciter les modules SkLearn qui permettent d'utiliser le successive having**

Scikit-learn propose deux classes principales permettant d'utiliser le Successive Halving : HalvingGridSearchCV et HalvingRandomSearchCV, disponibles dans le module sklearn.model_selection. La première applique le Successive Halving à une grille prédéfinie de paramètres, tandis que la seconde sélectionne aléatoirement les configurations à tester. Ces fonctionnalités étant encore expérimentales, elles doivent être activées au préalable avec sklearn.experimental.enable_halving_search_cv.

**§ Lister les paramètres et expliquer leur intérêt**

Les paramètres essentiels du Successive Halving sont factor, qui contrôle la proportion de candidats conservés à chaque étape, resource, qui définit la ressource augmentée progressivement, et min_resources / max_resources, qui déterminent la quantité minimale et maximale de ressources attribuées aux candidats. Les autres paramètres comme cv, scoring et n_jobs permettent respectivement de contrôler la validation croisée, la métrique d'évaluation et la parallélisation des calculs.

**§ Proposer un protocole qui permet de comparer les méthodes GridSearch et RandomSearch avec et sans le successive halving**

## Etape 1 Fixer le modèle et les données

Comme pour le TP4 on utilise RandomForestClassifier()

## Etape 2 Définir l'espace d'hyperparamètres

```py
PARAM_GRID = {
    "n_estimators": [100, 200, 300, 400, 500, 600],
    "max_depth": [5, 10, 15, 20, None],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False],
}
```

## Etape 3 Fixer la CV

Toujours comme le TP4 on garde une CV à 5 plis, il n'y aura rien à faire c'est la valeur par défault. 

## Etape 4 Déffinition des paramètres

`factor` &rarr; 2 \
`n_samples` &rarr; 'n_samples' \

## Etape 5 Mesurer plusieurs critères

On sauvegarde ces données : 
- meilleur score de validation croisée
- temps total de recherche
- meilleurs hyperparamètres trouvés

## Pour la suite 

Pour le randomSearch on fixe également le `n_iter` à 50. 

**§ Appliquer ce protocole sur le jeu de données digits**

In [10]:
import sklearn
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, HalvingRandomSearchCV
import numpy as np

j_digits = load_digits()
X, y = j_digits.data, j_digits.target

clf = RandomForestClassifier(random_state=0)

PARAM_GRID = {
    "n_estimators": [100, 200, 300, 400, 500, 600],
    "max_depth": [5, 10, 15, 20, None],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False],
}

search = HalvingGridSearchCV(clf, param_grid=PARAM_GRID, random_state=0, n_jobs=-1, factor=2).fit(X, y)

print(search.best_score_, search.best_params_)

0.9480544670846396 {'bootstrap': False, 'max_depth': 20, 'max_features': 'log2', 'n_estimators': 500}


In [13]:
rng = np.random.default_rng(0)

n_estimators = np.clip(
    rng.normal(loc=500, scale=200, size=1000).astype(int),
    10,
    1500
)

max_depth = np.clip(
    rng.normal(loc=10, scale=4, size=1000).astype(int),
    1,
    30
)

PARAM_RANDOM = {
    "n_estimators": n_estimators,
    "max_depth": max_depth,
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}


search = HalvingRandomSearchCV(clf, param_distributions=PARAM_GRID, random_state=0, n_jobs=-1, factor=2).fit(X, y)

print(search.best_score_, search.best_params_)

0.9474255485893417 {'n_estimators': 600, 'max_features': 'log2', 'max_depth': 15, 'bootstrap': False}


**§ Discuter les résultats obtenus**

### AutoML 

L'auto-ML permet d'automatiser le choix des composants et leur paramétrage d'un pipeline de machine learning.

**§ Rappeler la définition d'un pipeline de machine learning**

[Auto-sklearn](https://www.automl.org/automl/auto-sklearn/) est un outil d'AutoML. 

**§ Définir un protocole expérimental permettant de trouver un pipeline (ici, normalisation + classification supervisée) performant sur le jeu de données breast-cancer**